# Chapter 02-08 · A full exploratory analysis, and the document it produces

**Label:** Applied  |  **Time:** ~90 minutes  |  **Difficulty:** moderate - no new ideas, seven
chapters of old ones applied at once to data that did not come from this course

**Prerequisites:** all of module 02 (02-01 to 02-07).

**Position in the learning path:** module 02, chapter 8 of 8 - the applied chapter that closes the
module. After this: the module 02 assessment, then **03-01** and the mathematics you need.

---

## Why this matters

Every chapter in this module taught one habit against one small, controlled example. Real data
arrives with all of the problems at once, none of them labelled, and a deadline.

This chapter works a **real dataset** - 20,640 rows from the 1990 United States census, distributed
with scikit-learn - from first load to a finished document. Nothing is planted. Everything found
below is genuinely in the file, and was found by asking this module's questions in order.

The dataset also has a property worth meeting early: **it is spotlessly clean**. No missing values,
no duplicate rows, every column numeric and correctly typed. It is also, as you are about to see,
misleading in at least four separate ways. Clean and trustworthy are unrelated properties.

## What you will be able to do

- Work an unfamiliar dataset in a fixed order rather than poking at it
- Recover a quantity the table does not contain, and use it to explain values that look impossible
- Find a ceiling in a variable, and work out who is sitting under it
- Show that 0.38% of the rows were hiding two real relationships
- Write the data dictionary and limitations document that the next person actually needs

## The deliverable

An EDA is not a set of charts. It is **a document plus a decision**: here is what each column means,
here is what is wrong with it, here is what you may and may not conclude. The chapter ends by
writing that document to a file.

## Warm-up: retrieve the whole module

From memory, one sentence each:

1. What question does 02-01 insist you answer before anything else?
2. What did the firmware update in 02-02 do to the night-time undocking counts, and why was the
   total misleading?
3. What kind of missingness does mean-imputation quietly make worse?
4. Why did the three-sigma rule delete every festival day?
5. What can exploratory analysis never settle, no matter how carefully it is done?

<br>

*Answers: (1) what does one row represent. (2) it raised them by about 1000% - a recording change,
not a behaviour change; the total moved only 27% and hid it. (3) not-at-random missingness, where
the missing values differ systematically from the observed ones - imputing the observed mean pulls
them towards a centre they never belonged to. (4) the standard deviation was inflated by the very
points being tested, and the rule assumes a single normal population when there were two. (5) the
direction of a causal arrow, and whether a relationship found by searching is real.*

## The situation

A colleague sends you a file and one line of description: *"California housing data - see if there
is anything useful in it for the pricing model."*

That is the whole brief, and it is a realistic one. Before touching the file, the questions from this
module give you an order to work in:

| # | Question | Chapter |
|---|---|---|
| 1 | Where did this come from, who collected it, when, and under what terms? | 02-02 |
| 2 | What does one row represent? | 02-01 |
| 3 | What is the shape, and what types are the columns? | 01-04 |
| 4 | What is missing, duplicated, or impossible? | 02-04 |
| 5 | What does each column's distribution look like, and does anything have a floor or ceiling? | 02-05 |
| 6 | Who is *not* in this data? | 02-03 |
| 7 | What moves with what, and at which level of aggregation? | 02-06 |
| 8 | What can I honestly claim, and what would change my mind? | 02-07 |

Work them in order. Steps 1 and 2 are the ones people skip, and they are the ones that decide whether
everything after is worth anything.

## Step 1 · Provenance, before the data

This dataset ships inside scikit-learn, which means somebody has already written down where it came
from. Read that first. It takes two minutes and it is the highest-value two minutes of the whole
exercise.

In [ ]:
from sklearn.datasets import fetch_california_housing

# NOTE: the first call downloads about 360 KB and caches it in ~/scikit_learn_data.
# Later calls are offline. The raw file is NOT committed to this repository.
bunch = fetch_california_housing(as_frame=True)
housing = bunch.frame

print(bunch.DESCR[:1500])

### What the documentation says, and what it does not

**What it tells us, and we can therefore write down:**

- **Source:** StatLib at Carnegie Mellon, `https://lib.stat.cmu.edu/datasets/houses.zip`, from the
  paper Pace & Barry, *Sparse Spatial Autoregressions*, Statistics and Probability Letters 33 (1997).
- **Origin:** derived from the **1990 United States census**.
- **Unit of observation:** one row per **census block group** - the smallest area for which the
  Census Bureau publishes sample data, typically 600 to 3,000 people.
- **Target:** median house value for the block group, **in hundreds of thousands of dollars**.
- **A warning, from the documentation itself:** rooms and bedrooms are given *per household*, so
  block groups with few households and many empty houses can take "surprisingly large values".

**What it does not tell us, which we must record as unknown rather than guess:**

- **No licence is stated.** The underlying census data are a work of the US federal government and
  the 1990 census aggregates are public, but the file itself arrives with no terms attached. For
  coursework this is fine; before shipping anything built on it, somebody has to check.
- **Nothing about how "median house value" was measured** - self-reported by householders, assessed,
  or from sale prices. The census long form asked owners to estimate their home's value, but the
  documentation here does not say so, so we record the question rather than the answer.
- **No collection dates finer than "1990"**, so nothing about seasonality or within-year change is
  available.

**And the fact that dominates everything else: this data is from 1990.** It describes a housing
market that no longer exists, in a country whose residential geography was shaped by policies -
including federally sanctioned mortgage redlining, which operated into the late 1960s and whose
boundaries still track neighbourhood outcomes decades later - that are visible in exactly the columns
here: location, income, and value. That does not make the dataset unusable. It makes "what is this
model *for*" a question you answer before you build it, not after. It goes in the limitations
section, and module 13 returns to it properly.

## Step 2 · Shape, types, and the things that are not wrong

Before looking for problems, establish the baseline: how big, what types, what is missing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("shape:", housing.shape)
print()
print(housing.dtypes.to_string())
print()
print("missing values per column :", int(housing.isna().sum().sum()))
print("exact duplicate rows      :", int(housing.duplicated().sum()))
print("rows where bedrooms exceed rooms:",
      int((housing["AveBedrms"] > housing["AveRooms"]).sum()))

In [ ]:
pd.set_option("display.width", 130)
print(housing.describe().T.round(3).to_string())

### Read the summary table before reading on

Nine numeric columns, 20,640 rows, **zero missing values and zero duplicate rows**. On a data-quality
checklist this file scores full marks.

Now look at the table again, slowly, with 02-05's question - *what has a floor or a ceiling, and what
has a tail?* Four things in that output should stop you:

1. `AveRooms` has a maximum of **141.909**. An average of 142 rooms per household.
2. `AveOccup` has a maximum of **1243.333** people per household, with a mean of 3.07.
3. `MedHouseVal` has a maximum of exactly **5.00001**, which is a suspiciously precise round number
   for a maximum.
4. `HouseAge` has a maximum of exactly **52**, and `MedInc` a maximum of **15.0001**.

None of these is a missing value, a duplicate, or a type error. A quality checklist finds none of
them. They are the four things wrong with this dataset, and every one of them is visible in a table
you can print in one line - if you know what to look at.

## Step 3 · What does one row represent?

The documentation says a block group. That has consequences the documentation does not spell out,
and they are the same consequences 02-01 warned about.

**A row is not a house.** It is an area containing hundreds of houses. Every "value" in the row is a
summary over those houses:

- `MedHouseVal` is a **median over the houses in the block group**. The dataset's own target is
  already an aggregate, so a model fitted here predicts *the median of an area*, never the price of
  a house. Anyone who deploys it as a house-price estimator has made 02-01's mistake at scale.
- `MedInc` is a median over **households**, not over people.
- `AveRooms` and `AveOccup` are averages over **households** - a third unit, different from both
  houses and people.

Three different denominators in one row. That is worth writing down, and it is also the key to the
absurd values.

### Recovering a column that is not there

`Population` is a count of people. `AveOccup` is people per household. If those two mean what they
say, then dividing one by the other returns **the number of households** - a column nobody supplied.

If the result comes out as whole numbers, the interpretation is confirmed. If it comes out ragged,
one of the definitions is wrong.

In [ ]:
households = housing["Population"] / housing["AveOccup"]
distance_from_integer = (households - households.round()).abs()

print("largest distance from a whole number : %.8f" % distance_from_integer.max())
print("share within 0.01 of a whole number  : %.4f" % (distance_from_integer < 0.01).mean())
print()
print("households per block group:  min %.0f   median %.0f   max %.0f"
      % (households.min(), households.median(), households.max()))
print("block groups with fewer than 20 households : %d" % (households.round() < 20).sum())
print("block groups with fewer than 50 households : %d" % (households.round() < 50).sum())

**Every single one is a whole number, to eight decimal places.** The definitions are confirmed, and
the dataset now has a column it did not ship with.

This is worth pausing on, because it is one of the most useful moves in exploratory work: a claimed
definition implies an arithmetic consequence, you check the consequence, and it either holds exactly
or it does not. Here it held exactly, which converts "the documentation says" into "I verified".

And it immediately explains problem 1 and 2. The median block group has **409 households**, but 119
of them have fewer than 20. An "average" over 6 households is not a summary of anything.

## Step 4 · The impossible averages

Look directly at the rows with the largest `AveOccup`, with the recovered household count beside
them.

In [ ]:
work = housing.copy()
work["households"] = households.round().astype(int)

extreme = work.nlargest(5, "AveOccup")[
    ["Population", "households", "AveOccup", "AveRooms", "MedHouseVal", "Latitude", "Longitude"]
]
print(extreme.round(2).to_string())

### Diagnosis

The worst row is a block group with **6 households and 7,460 residents**. That is not a data error -
`Population / households` is exactly 1243.33, so the arithmetic is internally consistent. It is a
**definition mismatch**: population counts everybody living in the area, including people in what
the census calls group quarters - dormitories, barracks, prisons, care homes - while `households`
counts private homes. Divide one by the other and you get a number that is arithmetically correct
and semantically meaningless.

The scikit-learn documentation attributes large values to vacation resorts with many empty houses.
That explains a large `AveRooms`; it does not obviously explain 7,460 people sharing six households,
which looks more like institutional housing. **Neither explanation is settled by the data**, and this
is 02-05's rule arriving on cue: whether an extreme value is an error, a rare real case, or a
different population is a question the data cannot answer. The coordinates are in the table - the
honest next step is to look them up, not to guess.

What we *can* settle from the data is the shape of the problem: the extreme ratios sit where the
denominator is small.

In [ ]:
suspect = (work["AveRooms"] > 20) | (work["AveOccup"] > 20)
print("rows with AveRooms > 20 or AveOccup > 20 : %d  (%.2f%% of the data)"
      % (suspect.sum(), 100 * suspect.mean()))
print()
print("median households in those rows    : %.0f" % work.loc[suspect, "households"].median())
print("median households everywhere else  : %.0f" % work.loc[~suspect, "households"].median())

### What those 79 rows were doing to the analysis

They are 0.38% of the data. Here is what happens to the relationship between two columns and the
target when they are set aside - not deleted from the file, set aside for this calculation.

In [ ]:
sane = work.loc[~suspect]

comparison = pd.DataFrame({
    "column": ["AveRooms", "AveOccup"],
    "r with target, all 20,640 rows": [
        round(float(work["AveRooms"].corr(work["MedHouseVal"])), 3),
        round(float(work["AveOccup"].corr(work["MedHouseVal"])), 3),
    ],
    "r with target, 20,561 sane rows": [
        round(float(sane["AveRooms"].corr(sane["MedHouseVal"])), 3),
        round(float(sane["AveOccup"].corr(sane["MedHouseVal"])), 3),
    ],
})
print(comparison.to_string(index=False))

`AveOccup` goes from **-0.024 - indistinguishable from nothing - to -0.242**. `AveRooms` goes from
0.152 to 0.274, nearly doubling.

Seventy-nine rows out of twenty thousand were suppressing two real relationships. Anyone who ranked
these columns by correlation and dropped the weak ones would have thrown away `AveOccup`, one of the
more informative columns in the file, on the strength of a number produced by six households in one
block group at 38.32 N, 121.98 W.

**Note carefully what was *not* done here.** No automatic rule was applied. 02-05's three-sigma
filter would have removed festival days along with errors; here, the extremes were understood first -
by recovering the household count and finding the tiny denominators - and only then set aside, with
the reason recorded. That order matters:

1. Find the extreme values.
2. Work out the mechanism that produces them.
3. Decide what to do, and write down the decision and its reason.

Removing them at step 1 is the mistake. Keeping them in silently is also a mistake - it is the one
that cost `AveOccup` its signal.

## Step 5 · The ceilings

Three columns had suspiciously round maxima. The way to check a ceiling is not to look at the
maximum, but to count how many rows are sitting exactly on it - a real maximum is one row, a ceiling
is a crowd.

In [ ]:
for column in ["MedHouseVal", "HouseAge", "MedInc"]:
    top = housing[column].max()
    at_top = (housing[column] == top).sum()
    print("%-12s max %9.5f   rows exactly at max: %5d  (%.2f%%)"
          % (column, top, at_top, 100 * at_top / len(housing)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, column in zip(axes, ["MedHouseVal", "HouseAge", "MedInc"]):
    ax.hist(housing[column], bins=60, color="#0072B2")
    ax.set_title(column)
    ax.set_xlabel(column)
axes[0].set_ylabel("block groups")
plt.tight_layout()
plt.show()

### Diagnosis: the data was clipped before it reached you

Three spikes, all at the right-hand edge. **965 block groups** - 4.68% - have a median house value of
exactly $500,001, and 1,273 have a house age of exactly 52 years. Values above those points were not
missing; they were **recorded as the maximum**. This is censoring, and 02-03 met it as the delivery
times that stopped at 30 minutes.

The consequences are specific:

- The target's true range is unknown above $500,000. A model trained here cannot predict above the
  cap, because it has never seen a number above it. On the most expensive areas in California it will
  be wrong, and confidently.
- `HouseAge` of 52 means "52 or older". Since the data is from 1990, that is "built in 1938 or
  earlier" - a category, not a number, but stored as a number and therefore silently treated as one
  by every model you fit.
- Any average of the target is an **underestimate**, because the largest values have been pulled
  down to the cap.

### Who is sitting at the ceiling?

02-04's question. If the capped rows were a random sample of the data, the cap would cost precision
and nothing else. They are not.

In [ ]:
capped = housing["MedHouseVal"] == housing["MedHouseVal"].max()

profile = pd.DataFrame({
    "at the $500k cap": housing.loc[capped].mean(numeric_only=True),
    "everything else": housing.loc[~capped].mean(numeric_only=True),
}).round(3)
print(profile.to_string())

Mean median-income of **7.83 against 3.68** - more than double. The censored rows are the richest
block groups in the state, which is exactly what you would expect and exactly what makes the
censoring dangerous: it is not-at-random in the sense of 02-04, and it removes information
specifically from the top of the range.

A concrete consequence, on a one-column fit:

In [ ]:
from sklearn.linear_model import LinearRegression

X = housing[["MedInc"]]
y = housing["MedHouseVal"]

all_rows = LinearRegression().fit(X, y)
uncapped_only = LinearRegression().fit(X.loc[~capped], y.loc[~capped])

print("slope fitted on all rows       : %.4f" % all_rows.coef_[0])
print("slope fitted excluding the cap : %.4f" % uncapped_only.coef_[0])
print()
print("Read in units: each extra $10,000 of median income is associated with")
print("  $%s of median house value using all rows," % format(round(all_rows.coef_[0] * 100000), ","))
print("  $%s once the censored rows are set aside." % format(round(uncapped_only.coef_[0] * 100000), ","))

The two slopes differ by about 4.3%. That is smaller than the correlation shifts in step 4, and the
honest report says so: **the censoring is a real problem for predictions near the top of the range
and a modest one for the overall slope**. Not every defect is a catastrophe, and an EDA that treats
them all as catastrophes is as useless as one that finds none.

Neither fit above is a model of anything - both are fitted on all the data with no holdout, which is
the mistake 00-01's memoriser made. They are here to size a defect, not to predict. Module 04 does
this properly.

## Step 6 · What moves with what

Only now - after the unit is understood, the impossible values explained, and the ceilings found - is
it worth looking at relationships. Correlations computed before step 4 would have been wrong about
two columns.

In [ ]:
correlations = sane.drop(columns=["households"]).corr(numeric_only=True)["MedHouseVal"]
print("Correlation with median house value (79 suspect rows set aside):")
print(correlations.drop("MedHouseVal").sort_values(ascending=False).round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
points = ax.scatter(sane["Longitude"], sane["Latitude"], c=sane["MedHouseVal"],
                    s=3, alpha=0.5, cmap="viridis")
fig.colorbar(points, ax=ax, label="median house value ($100,000s)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Every block group, coloured by value")
plt.tight_layout()
plt.show()

### Reading it honestly

`MedInc` at **0.689** is far and away the strongest single relationship, and nothing else exceeds
0.3. The map shows the rest of the story: value is concentrated on the coast, around two metropolitan
areas, with the interior uniformly cheaper.

Three cautions before any of that goes on a slide, one from each of the last three chapters:

- **02-06.** These correlations are computed across the whole state, pooling San Francisco with rural
  Kern County. A within-city relationship can differ from the pooled one, in strength or in sign.
  "Older houses are worth more" statewide may be "older houses are worth less" inside any given city,
  if the old housing stock happens to sit in the expensive cities. This dataset has no city column,
  so the check requires building one from the coordinates - a real task, and exercise E7.
- **02-07.** Every number here is an association. `MedInc` and value move together; nothing in this
  file says which way the arrow points, and the plausible answer is that both are driven by
  desirability of location. Nobody raises a neighbourhood's house prices by giving its residents a
  pay rise.
- **02-07 again.** Eight columns were screened against one target. That is eight looks, so the
  benchmark for "the largest correlation among unrelated columns" is not zero. The noise floor
  scales as 1/sqrt(n), which on 20,640 rows is about 0.007, so eight looks put the benchmark near
  0.02 and 0.689 is not a fluke. **The benchmark is low here because the sample is enormous**, which
  is a fact about this dataset and not a general licence.

## Step 7 · Who is not in this data

02-03's question, and the one this dataset answers worst.

- **No renters' costs.** Every column is about owned housing and its value. A block group of renters
  contributes a median house value for whatever owned homes exist there.
- **Vacant housing is invisible** except through its effect on the per-household averages.
- **The people are gone.** The census recorded race, household composition and tenure; this file kept
  income, rooms and location. That is a choice somebody made in 1997, and it means questions about
  who lives where cannot be asked of this file - which is a limitation, not a protection.
- **1990 California only.** Nothing here transfers to another state or another decade without an
  argument, and the argument would have to be about housing markets, not about statistics.

None of this is discoverable from the file. It comes from knowing what a census contains and
comparing it to what arrived - which is why step 1 came first.

## The deliverable · a data dictionary

Everything above is now written down, per column, in the form the next person needs: what it means,
its unit, its range, and what is wrong with it. The last column is the one that makes this document
worth writing.

In [ ]:
dictionary = pd.DataFrame([
    ("MedInc", "median household income in the block group", "$10,000s", "0.50 - 15.00",
     "CEILING at 15.0001 (49 rows). Median over households, not people."),
    ("HouseAge", "median age of houses in the block group", "years, in 1990", "1 - 52",
     "CEILING at 52 (1,273 rows) = built 1938 or earlier. Ordinal above the cap."),
    ("AveRooms", "mean rooms per household", "rooms", "0.85 - 141.91",
     "Ratio with a tiny denominator in small block groups. 69 rows above 20."),
    ("AveBedrms", "mean bedrooms per household", "rooms", "0.33 - 34.07",
     "Same ratio problem. Never exceeds AveRooms - checked, 0 violations."),
    ("Population", "residents of the block group", "people", "3 - 35,682",
     "Includes group quarters, which have no households. 104 rows below 50."),
    ("AveOccup", "mean residents per household", "people", "0.69 - 1243.33",
     "Population / households, so group quarters break it. 10 rows above 20."),
    ("Latitude", "block group centroid", "decimal degrees", "32.54 - 41.95", "California only."),
    ("Longitude", "block group centroid", "decimal degrees", "-124.35 - -114.31", "California only."),
    ("MedHouseVal", "TARGET: median house value in the block group", "$100,000s", "0.15 - 5.00001",
     "CENSORED at $500,001 (965 rows, 4.68%). Censored rows are the richest."),
], columns=["column", "meaning", "unit", "range", "caution"])

pd.set_option("display.max_colwidth", 70)
print(dictionary.to_string(index=False))

In [ ]:
from pathlib import Path

limitations = '''
## Limitations - California housing (1990)

1. UNIT OF OBSERVATION. One row is a census block group (typically 600-3,000 people), not a
   house and not a household. The target is already a median over houses. A model fitted here
   predicts an area's median, never a house's price. Deploying it as a house valuer is a
   category error.
2. THREE DENOMINATORS. Values are per block group (Population), per household (AveRooms,
   AveBedrms, AveOccup, MedInc) or per house (MedHouseVal). They are not interchangeable.
3. CENSORED TARGET. 965 rows (4.68%) sit exactly at $500,001. The true values above that are
   unknown, the affected rows are the richest block groups (mean MedInc 7.83 vs 3.68), and no
   model trained on this data can predict above the cap. Any mean of the target is an
   underestimate.
4. CENSORED FEATURES. HouseAge caps at 52 (1,273 rows); MedInc caps at 15.0001 (49 rows).
5. RATIO ARTEFACTS. 79 rows have AveRooms > 20 or AveOccup > 20, caused by block groups with
   very few households, or by group quarters where residents have no household. Leaving them
   in suppresses two real relationships: r(AveOccup, target) is -0.024 with them and -0.242
   without; r(AveRooms, target) is 0.152 with and 0.274 without.
6. NO CITY OR REGION COLUMN. All reported correlations are pooled across the whole state and
   may differ within a city, in size or in sign. Not checked here.
7. AGE AND SCOPE. 1990 census, California only. It describes a market that no longer exists.
   Geographic patterns in US housing value of this era reflect historical policy including
   mortgage redlining; any use that treats the patterns as neutral facts inherits that.
8. LICENCE NOT STATED. Source is StatLib (Pace & Barry 1997), derived from the 1990 US census.
   No licence accompanies the file. Check before any use outside coursework.
9. MEASUREMENT UNKNOWN. The documentation does not say how house value was determined -
   self-reported, assessed, or from sales. This affects how much to trust the target.

## What may be claimed from this data

- Median income is strongly associated with median house value across California block groups
  (r = 0.69). Location is associated with both.
- Nothing about causation. Nothing about individual houses. Nothing about 2026.
'''

def to_markdown_table(df):
    header = "| " + " | ".join(df.columns) + " |"
    rule = "|" + "---|" * len(df.columns)
    rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([header, rule] + rows)


report = ("# California housing - data dictionary\n\n"
          + to_markdown_table(dictionary) + "\n" + limitations)
output_path = Path("data_dictionary_draft.md")
output_path.write_text(report)

print("wrote %s  (%d characters, %d lines)"
      % (output_path, len(report), report.count("\n")))
print()
print(limitations[:700])

### Why the limitations section is the valuable half

The column table can be reconstructed by anyone with the file and an hour. The limitations cannot -
they are the output of the work, and every one of them corresponds to a specific mistake the next
person is now not going to make.

Notice the shape of the entries. Each one says **what is wrong, how big it is, and what it stops you
concluding**. "The target is censored" is a note. "965 rows are at the cap, they are the richest
block groups, and no model trained here can predict above $500,000" is a limitation somebody can act
on.

Notice also entry 6: *not checked here*. A limitations section that only lists what you found implies
you looked everywhere. Recording what you did not check is what makes the rest of it trustworthy.

## Common misconceptions

**"No missing values and no duplicates means the data is good."**
This dataset has neither and is misleading in four separate ways. Those checks find *storage*
problems. Ceilings, unit mismatches, ratio artefacts and censoring are *meaning* problems, and no
automated check finds them because they require knowing what the numbers are supposed to be.

**"The extreme values are errors, so remove them."**
The 79 extreme rows are arithmetically correct - every one satisfies `AveOccup = Population /
households` exactly. They are a definition mismatch, not a mistake, and the reason to set them aside
is that mismatch, not their size. Remove-then-ask is how 02-05's festivals were lost.

**"Correlation ranking is a reasonable way to shortlist columns."**
`AveOccup` correlates at -0.024 with the target, which looks like nothing, and at -0.242 once 0.38%
of rows are handled. Ranking before cleaning ranks the artefacts.

**"EDA is the charts."**
The charts are the cheapest part. What survives this chapter is a document; the histograms served
their purpose the moment they revealed three spikes.

**"The data dictionary is documentation, so it can be written at the end."**
It *is* the analysis. Every row of that table is a finding that took a specific check to establish -
which is why writing it reveals the columns you never actually looked at.

## Exercises

Solutions: `solutions/02_data_literacy/02-08_applied_eda_solutions.ipynb`.

### Quick understanding

**E1.** This dataset has no missing values and no duplicates. Name three defects it has anyway, and
say which module 02 chapter each one belongs to.

**E2.** Why does a row with `AveOccup = 1243` not count as a data-entry error?

**E3.** What exactly does `HouseAge = 52` mean, and why is storing it as the number 52 a problem?

### Hand calculation

**E4.** A block group has `Population = 1200` and `AveOccup = 2.5`. How many households? If
`AveRooms = 5.2`, how many rooms in total? Now suppose the true population is 1,200 but 900 of them
live in a college dormitory. What is `AveOccup` actually measuring?

**E5.** 965 of 20,640 rows are at the $500,001 cap, and those rows would truly have averaged, say,
$720,000. By roughly how much does the reported mean target understate the truth? (Work in
$100,000s, use the printed means, and say why your answer is a lower bound.)

### Coding

**E6.** Write `ceiling_check(df, column)` that reports the maximum, how many rows equal it, what
share that is, and how many equal the second-largest value - flagging a likely ceiling when the top
value holds more than 1% of rows. Run it on all nine columns and report which ones flag.

**E7.** The dataset has no city column. Build a crude one: cluster the block groups on latitude and
longitude into 8 groups with `KMeans`, then compute `r(HouseAge, MedHouseVal)` overall and within
each cluster. Does the relationship hold its sign everywhere? Relate what you find to 02-06.

**E8.** Set aside the 79 suspect rows, then recompute the full correlation matrix with the target
before and after. Which column's correlation moves most in absolute terms, and which changes rank
the most?

### Interpretation

**E9.** Your colleague fits a model on this data and reports a mean absolute error of $52,000, then
proposes using it to price individual houses. Give three separate reasons to refuse, ordered by how
badly each one breaks the proposal.

**E10.** The capped rows have mean `MedInc` of 7.83 against 3.68 elsewhere. Explain, in two
sentences, why that makes the censoring worse than if the capped rows had been a random 4.68%.

### Debugging

**E11.** An analyst standardises every column, runs a three-sigma outlier filter across the whole
frame, and reports that it removed 1,142 rows including "all the bad ones". Name two things wrong
with this procedure as applied to *this* dataset specifically.

### Exam and interview reasoning

**E12.** "Walk me through how you'd approach a dataset you've never seen." Answer in eight steps
using this chapter's order, one sentence each. Then say which step people skip and what it costs.

### Transfer to a different situation

**E13.** You are handed hospital admissions data: one row per ward per day, with average length of
stay, average patient age, and bed count. Name the equivalent of each of this chapter's four defects
that you would look for first, and the single arithmetic check you would run to confirm the unit of
observation.

### Explain it to someone non-technical

**E14.** Your manager asks: "It's got twenty thousand rows and nothing's missing - why did this take
you a day?" Answer in under 120 words.

### Optional challenge

**E15.** Write the whole thing up as if handing over: a one-page summary with the four defects, their
sizes, the three columns you would not use without treatment, and a recommendation on whether the
dataset supports the pricing model your colleague asked about. Then write the one-sentence version
for someone who will not read the page.

In [ ]:
# Your workspace. In memory: housing, work, sane, households, capped, suspect,
# dictionary, bunch (bunch.DESCR has the full documentation).

## Mastery check

- [ ] Work an unfamiliar dataset in a defensible order, and say why provenance comes first
- [ ] Detect a ceiling by counting rows at the maximum rather than by looking at the maximum
- [ ] Check who sits under a ceiling, and say why it matters that they are not random
- [ ] Recover an implied column and use it to explain values that look impossible
- [ ] Explain why 0.38% of rows changed two correlations, without calling those rows errors
- [ ] Write a limitations section where each entry names a conclusion it forbids

## What should now feel instinctive

- Reading the documentation before the data, and writing down what it does *not* say
- Seeing a suspiciously round maximum and immediately counting the rows sitting on it
- Asking what the denominator is, every time you see an average
- Understanding an extreme value before deciding what to do with it
- Finishing an analysis with a document rather than a folder of charts
- Recording what you did not check

## Flashcards

| Front | Back |
|---|---|
| No missing values, no duplicates | Storage is fine. Says nothing about meaning |
| Suspiciously round maximum | Count rows at that value: one is a maximum, a crowd is a ceiling |
| Censored target | Cannot predict above the cap; means are underestimates; check who is up there |
| An average in a row | Ask what the denominator is - and whether it is ever tiny |
| Ratio with a small denominator | Produces extreme values that are arithmetically correct and meaningless |
| Extreme values, correct order | Find, explain, then decide - never decide first |
| California housing target | Median over houses in a block group, capped at $500,001, from 1990 |
| Best single predictor here | MedInc, r = 0.689; everything else below 0.3 |
| The EDA deliverable | Data dictionary plus limitations, each limitation naming what it forbids |

## Next

**Module 02 is complete.** Eight chapters: what a row is, where data comes from, who is missing, what
is broken, how values are distributed, how columns relate, how to report honestly, and how to do all
of it at once on real data.

The **module 02 assessment** in `assessments/` comes next - it is cumulative and worked without
notes. After that, **03-01** starts the mathematics: the small, specific amount you need, introduced
where it is used, beginning with the notation that makes everything after it readable.